# Test SFT sur dataset medical

### DATASET

- Load dataset
- Preprocessing pour avoir le format (prompt/completion) nécessaire pour SFTTrainer

In [ ]:
from pathlib import Path
from datasets import load_dataset, load_from_disk

# utility code - télécharge le dataset si pas déjà présent sur le disque
# et le charge localement

HF_DATASET_ID = "FreedomIntelligence/medical-o1-reasoning-SFT"
HF_CONFIG = "en"
DATASET_DIR = Path("../datasets").resolve()

# Ensure target directory exists
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# "Empty" means no files/folders inside
is_empty = not any(DATASET_DIR.iterdir())

if is_empty:
    print(f"{DATASET_DIR} is empty -> downloading from Hub and saving to disk...")
    dataset = load_dataset(HF_DATASET_ID, HF_CONFIG)
    dataset.save_to_disk(str(DATASET_DIR))
else:
    print(f"{DATASET_DIR} is not empty -> loading from disk...")
    dataset = load_from_disk(str(DATASET_DIR))

print(dataset)

In [ ]:
from pprint import pprint

def preprocess_function(example):
    return {
        "prompt": [{"role": "user", "content": example["Question"]}],
        "completion": [
            {"role": "assistant", "content": f"<think>{example['Complex_CoT']}</think>{example['Response']}"}
        ],
    }

dataset = dataset.map(preprocess_function, remove_columns=["Question", "Response", "Complex_CoT"])
pprint(next(iter(dataset["train"])))

In [ ]:
# subset first, then split
N_MAX = 100
SPLIT_SEED = 42
VAL_RATIO = 0.1

base_train = dataset["train"]

if N_MAX <= 0:
    raise ValueError("N_MAX must be > 0")

n_subset = min(N_MAX, len(base_train))
subset_train = base_train.shuffle(seed=SPLIT_SEED).select(range(n_subset))

split = subset_train.train_test_split(
    test_size=VAL_RATIO,
    seed=SPLIT_SEED,
    shuffle=True,
)

train_ds = split["train"]
val_ds = split["test"]

print(f"Original train size: {len(base_train)}")
print(f"Subset size:         {len(subset_train)}")
print(f"Train size:          {len(train_ds)}")
print(f"Validation size:     {len(val_ds)}")

### MODELE

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model
import torch

In [ ]:
# utility function pour calculer le nombre de paramètres trainables
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total

In [ ]:
# utility code - idem, télécharge le modèle si pas déjà présent sur le disque
# et le sauve localement

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
MODEL_DIR = Path("../models/qwen2_5_3b_instruct").resolve()

MODEL_DIR.mkdir(parents=True, exist_ok=True)
is_empty = not any(MODEL_DIR.iterdir())

if is_empty:
    print(f"{MODEL_DIR} is empty -> downloading model/tokenizer from Hub and saving to disk...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,   # ou torch.float16 selon ton GPU
        device_map="auto"
    )
    tokenizer.save_pretrained(str(MODEL_DIR))
    model.save_pretrained(str(MODEL_DIR))
else:
    print(f"{MODEL_DIR} is not empty -> loading model/tokenizer from disk...")
    tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
        str(MODEL_DIR),
        local_files_only=True,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

In [ ]:
# base_model=AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
base_model = model

# Before LoRA (full model)
full_trainable, full_total = count_params(base_model)

print(f"Full FT trainable: {full_trainable:,}")

In [ ]:
# using PEFT and LoRA pour réduire le besoin en VRAM

from peft import LoraConfig
RANG=4

peft_config=LoraConfig(
    task_type="CAUSAL_LM",   # decrit quel type de modèle on adapte
    inference_mode=False, # ordonne que les paramètres soient entrainables
    r=RANG, # rang !
    lora_alpha=2*RANG, # scaling factor pour l'update LoRA
    lora_dropout=0.05, # dropout pour le LoRA
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # LoRA sur les têtes d'attention
)

In [ ]:
lora_model=get_peft_model(base_model, peft_config)

# After LoRA wrapping
lora_trainable, lora_total = count_params(lora_model)

reduction_vs_full_ft = 100 * (1 - (lora_trainable / full_trainable))
trainable_pct_of_total = 100 * (lora_trainable / lora_total)

print(f"Full FT trainable: {full_trainable:,}")
print(f"LoRA trainable:    {lora_trainable:,}")
print(f"Reduction:         {reduction_vs_full_ft:.2f}%")
print(f"LoRA trainable %:  {trainable_pct_of_total:.4f}%")

### EVALUATION metrics

In [ ]:
import numpy as np

# Optional metric: token-level accuracy (ignores -100 labels)
def preprocess_logits_for_metrics(logits, labels):
    # logits est typiquement (batch, seq_len, vocab_size)
    if isinstance(logits, tuple):
        logits = logits[0]
    # renvoie (batch, seq_len) avec le predicted token id
    return logits.argmax(dim=-1)

def compute_metrics(eval_pred):
    preds, labels = eval_pred  # numpy arrays
    # next-token shift
    preds = preds[:, :-1].reshape(-1)
    labels = labels[:, 1:].reshape(-1)

    mask = labels != -100
    if mask.sum() == 0:
        return {"token_accuracy": 0.0}

    # compte 1 pour les bons tokens prédits, et la moyenne
    acc = (preds[mask] == labels[mask]).astype(np.float32).mean().item()
    return {"token_accuracy": acc}

In [ ]:
# 3) Enable eval during training
sft_args = SFTConfig(
    output_dir="outputs/sft-medical",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,      # important pour ma pauvre 3080 Ti 12 Go
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    num_train_epochs=3,
    
    # log each epoch
    logging_strategy="epoch",
    logging_dir="./logs",
    # logging_steps=10,
    save_steps=100,
    
    # Optional: use tensorboard for better visualization
    report_to=["tensorboard"],

    # max_seq_length=512,
    gradient_checkpointing=True,
    bf16=True,                         # or fp16=True on your hardware

    eval_strategy="steps",             # if your version expects evaluation_strategy, use that key instead
    eval_steps=100,
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    eval_accumulation_steps=8,         # helps eval memory
)

In [ ]:
trainer = SFTTrainer(
    model=lora_model,  # use the already reduced model
    args=sft_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    # processing_class=tokenizer
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics
)

In [ ]:
trainer.train()

### QUELQUES TESTS

In [ ]:
def parse_completion(completion_text):
    """
    Parse completion text to extract reasoning and final answer.
    Format: <think>REASONING</think>FINAL_ANSWER
    
    Returns: (reasoning_text, final_answer_text)
    """
    if "<think>" in completion_text and "</think>" in completion_text:
        start_idx = completion_text.find("<think>") + len("<think>")
        end_idx = completion_text.find("</think>")
        reasoning = completion_text[start_idx:end_idx].strip()
        final_answer = completion_text[end_idx + len("</think>"):].strip()
    else:
        # No think tags, treat entire text as final answer
        reasoning = ""
        final_answer = completion_text.strip()
    
    return reasoning, final_answer

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer

N_SAMPLES = 5
SAMPLE_SEED = 123
MAX_NEW_TOKENS = 256

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_for_eval = trainer.model
model_for_eval.eval()

rng = np.random.default_rng(SAMPLE_SEED)
n_eval = min(N_SAMPLES, len(val_ds))
sample_indices = rng.choice(len(val_ds), size=n_eval, replace=False).tolist()

print(f"Sampling {n_eval} examples from validation set of size {len(val_ds)}")

In [ ]:
results = []

for idx in sample_indices:
    ex = val_ds[int(idx)]

    # prompt already stored as chat messages
    messages = ex["prompt"]
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model_for_eval.device)

    with torch.no_grad():
        generated = model_for_eval.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    gen_tokens = generated[0][prompt_len:]
    pred_completion = tokenizer.decode(gen_tokens, skip_special_tokens=True)

    # Parse ground truth
    gt_full = ex["completion"][0]["content"]
    gt_reasoning, gt_final_answer = parse_completion(gt_full)

    # Parse prediction
    pred_reasoning, pred_final_answer = parse_completion(pred_completion)

    results.append(
        {
            "idx": int(idx),
            "prompt": messages[0]["content"],
            "gt_reasoning": gt_reasoning,
            "gt_final_answer": gt_final_answer,
            "pred_reasoning": pred_reasoning,
            "pred_final_answer": pred_final_answer,
            "gt_full": gt_full,
            "pred_full": pred_completion,
        }
    )

print(f"Built {len(results)} comparisons with parsed components.")

In [ ]:
preview_df = pd.DataFrame(results)[
    ["idx", "prompt", "gt_reasoning", "gt_final_answer", "pred_reasoning", "pred_final_answer"]
].copy()

# Truncate for display
preview_df["prompt"] = preview_df["prompt"].str.slice(0, 80) + "..."
preview_df["gt_reasoning"] = preview_df["gt_reasoning"].str.slice(0, 100) + "..."
preview_df["gt_final_answer"] = preview_df["gt_final_answer"].str.slice(0, 100) + "..."
preview_df["pred_reasoning"] = preview_df["pred_reasoning"].str.slice(0, 100) + "..."
preview_df["pred_final_answer"] = preview_df["pred_final_answer"].str.slice(0, 100) + "..."

preview_df

In [ ]:
for i, r in enumerate(results, start=1):
    print("=" * 120)
    print(f"Sample {i} | Validation index: {r['idx']}")
    print("\n[PROMPT]")
    print(r["prompt"])
    print("\n[GROUND TRUTH - Reasoning]")
    print(r["gt_reasoning"] if r["gt_reasoning"] else "(no reasoning tags)")
    print("\n[GROUND TRUTH - Final Answer]")
    print(r["gt_final_answer"])
    print("\n[PREDICTED - Reasoning]")
    print(r["pred_reasoning"] if r["pred_reasoning"] else "(no reasoning tags)")
    print("\n[PREDICTED - Final Answer]")
    print(r["pred_final_answer"])
    print()